In [4]:
# Mount Google Drive and load data preview
import os
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# Define paths
dataset_path = "/content/drive/MyDrive/ieee-fraud-detection (1)"
transaction_path = f"{dataset_path}/train_transaction.csv"

# Verify file presence and load sample
if os.path.exists(transaction_path):
    print("File located. Loading preview")
    cols_preview = ["TransactionID", "TransactionDT", "card1", "addr1", "D1", "D2", "isFraud"]
    df_sample = pd.read_csv(transaction_path, usecols=cols_preview, nrows=5)
    display(df_sample)
else:
    print(f"Error: File not found at {transaction_path}")
    print("Directory contents:", os.listdir(dataset_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File located. Loading preview


,TransactionID,isFraud,TransactionDT,card1,addr1,D1,D2
0,2987000,0,86400,13926,315.0,14.0,NaN
1,2987001,0,86401,2755,325.0,0.0,NaN
2,2987002,0,86469,4663,330.0,0.0,NaN
3,2987003,0,86499,18132,476.0,112.0,112.0
4,2987004,0,86506,4497,420.0,0.0,NaN


In [5]:
# Compare baseline entity candidates: card1 vs card1 + addr1

# Load required columns for the full dataset
cols = ["TransactionID", "card1", "addr1"]
df = pd.read_csv(transaction_path, usecols=cols)

# Create candidate identifiers
df["id_card1"] = df["card1"].astype(str)
df["id_card1_addr1"] = np.where(
    df["addr1"].isna(),
    np.nan,
    df["card1"].astype(str) + "_" + df["addr1"].astype(str)
)

# Function to calculate and print summary statistics
def get_stats(series, name):
    valid = series.dropna()
    counts = valid.value_counts()

    total = len(series)
    missing = series.isna().sum()
    unique = len(counts)
    singletons = (counts == 1).sum()
    repeaters = unique - singletons

    print(f"--- {name} ---")
    print(f"Missing records: {missing} ({(missing/total)*100:.2f}%)")
    print(f"Unique entities: {unique:,}")
    print(f"Entities with 1 transaction: {singletons:,} ({(singletons/unique)*100:.2f}%)")
    print(f"Entities with >1 transaction: {repeaters:,} ({(repeaters/unique)*100:.2f}%)")
    print(f"Max transactions by one entity: {counts.max():,}")
    print(f"Median transactions per entity: {counts.median()}\n")

get_stats(df["id_card1"], "Candidate 1: card1")
get_stats(df["id_card1_addr1"], "Candidate 2: card1 + addr1")

--- Candidate 1: card1 ---
Missing records: 0 (0.00%)
Unique entities: 13,553
Entities with 1 transaction: 3,444 (25.41%)
Entities with >1 transaction: 10,109 (74.59%)
Max transactions by one entity: 14,932
Median transactions per entity: 4.0

--- Candidate 2: card1 + addr1 ---
Missing records: 65706 (11.13%)
Unique entities: 37,531
Entities with 1 transaction: 14,818 (39.48%)
Entities with >1 transaction: 22,713 (60.52%)
Max transactions by one entity: 5,885
Median transactions per entity: 2.0



## Baseline Groupings Evaluation
Testing card1 and card1 + addr1 revealed that both candidate identifiers are too broad to represent individual users. The largest card1 entity contained nearly 15,000 transactions, and while adding the billing region (addr1) helped, the largest group still contained almost 6,000 transactions. Because these numbers are too high for a single user over six months, the next step is to extract an anchor date from the D features to split these massive groups.

In [6]:
# Calculate D1 anchor day and check stability within groups


# Load the necessary columns
cols = ["TransactionID", "TransactionDT", "card1", "addr1", "D1"]
df = pd.read_csv(transaction_path, usecols=cols)

# Recreate the card1 + addr1 candidate
df["id_card1_addr1"] = np.where(
    df["addr1"].isna(),
    np.nan,
    df["card1"].astype(str) + "_" + df["addr1"].astype(str)
)

# Calculate the estimated account start day (Anchor Day)
# TransactionDT is in seconds. 86400 seconds = 1 day.
df["D1_anchor"] = np.floor((df["TransactionDT"] / 86400) - df["D1"])

# Look at the top 5 largest card1 + addr1 groups
top_groups = df["id_card1_addr1"].value_counts().head(5).index

print("Unique D1 anchor days found in the top 5 largest card1+addr1 groups:\n")

for group_id in top_groups:
    group_data = df[df["id_card1_addr1"] == group_id]
    valid_anchors = group_data["D1_anchor"].dropna()
    unique_anchors = valid_anchors.nunique()
    total_tx = len(group_data)

    print(f"Group {group_id}: {total_tx} transactions")
    print(f"Number of unique underlying anchor days: {unique_anchors}\n")

Unique D1 anchor days found in the top 5 largest card1+addr1 groups:

Group 17188_299.0: 5885 transactions
Number of unique underlying anchor days: 499

Group 12695_325.0: 5774 transactions
Number of unique underlying anchor days: 504

Group 9500_204.0: 4665 transactions
Number of unique underlying anchor days: 420

Group 12839_264.0: 3547 transactions
Number of unique underlying anchor days: 391

Group 16132_299.0: 3537 transactions
Number of unique underlying anchor days: 403



## Validating Group Stability with D1
Evaluating the D1 feature (days since account creation) within the largest card1 + addr1 groups confirmed that they contain mixed users. The top five groups each had roughly 400 to 500 unique account creation dates. Since a single user should only have one account creation date, the next step is to combine card1, addr1, and the calculated D1 anchor date into a unified identifier to isolate actual individuals.

In [7]:
# Create and evaluate Candidate 3: card1 + addr1 + D1_anchor
cols = ["TransactionID", "TransactionDT", "card1", "addr1", "D1"]
df = pd.read_csv(transaction_path, usecols=cols)

# Calculate D1 anchor day
df["D1_anchor"] = np.floor((df["TransactionDT"] / 86400) - df["D1"])

# Combine into a single ID, handling missing values
df["id_combined"] = np.where(
    df["addr1"].isna() | df["D1"].isna(),
    np.nan,
    df["card1"].astype(str) + "_" +
    df["addr1"].astype(str) + "_" +
    df["D1_anchor"].astype(str)
)

# Run the stats function defined in Step 1
get_stats(df["id_combined"], "Candidate 3: card1 + addr1 + D1_anchor")

--- Candidate 3: card1 + addr1 + D1_anchor ---
Missing records: 66794 (11.31%)
Unique entities: 199,070
Entities with 1 transaction: 115,513 (58.03%)
Entities with >1 transaction: 83,557 (41.97%)
Max transactions by one entity: 1,414
Median transactions per entity: 1.0



## Evaluating the Combined Proxy ID
Combining card1, addr1, and the D1 anchor date drastically improved our user grouping. We generated nearly 200,000 unique entities, with a realistic median of 1 transaction per user. Over 40% of these entities occur multiple times, providing a massive pool of repeat behavior to build sequence features from. The maximum transactions for a single entity dropped to 1,414 (about 8 per day), which is vastly more realistic than the previous 14,000+, though it might still capture a few corporate cards or bots. The next step is to measure the timespan between the first and last transaction for these repeating entities to ensure they remain stable across the 6-month dataset.

In [8]:
# Measure the timespan of repeating entities
# df["id_combined"] and df["TransactionDT"] are already in memory

# Filter for entities that have more than 1 transaction
counts = df["id_combined"].value_counts()
repeat_ids = counts[counts > 1].index
df_repeaters = df[df["id_combined"].isin(repeat_ids)]

# Calculate the min and max TransactionDT for each entity
timespans = df_repeaters.groupby("id_combined")["TransactionDT"].agg(["min", "max"])

# Convert the difference from seconds to days
timespans["span_days"] = (timespans["max"] - timespans["min"]) / 86400

print("--- Timespan of Repeating Entities ---")
print(f"Total repeating entities analyzed: {len(timespans):,}")
print(f"Median timespan (days): {timespans['span_days'].median():.2f}")
print(f"Mean timespan (days): {timespans['span_days'].mean():.2f}")
print(f"Entities active for > 1 day:  {(timespans['span_days'] > 1).sum():,} ({((timespans['span_days'] > 1).sum() / len(timespans)) * 100:.2f}%)")
print(f"Entities active for > 30 days: {(timespans['span_days'] > 30).sum():,} ({((timespans['span_days'] > 30).sum() / len(timespans)) * 100:.2f}%)")

--- Timespan of Repeating Entities ---
Total repeating entities analyzed: 83,557
Median timespan (days): 19.08
Mean timespan (days): 42.58
Entities active for > 1 day:  56,801 (67.98%)
Entities active for > 30 days: 35,874 (42.93%)


## Measuring Entity Timespans
Analyzing the timespan of repeating entities shows strong long-term stability. The average repeating user is active for about 43 days, with nearly 68% spanning more than a single day and 43% remaining active for over a month. Because these entities reliably persist over time rather than just firing in isolated bursts, we have the evidence needed to make our final feasibility conclusion

In [9]:
# # Test whether proxy entities seen earlier reappear in a later time period

# df["day"] = np.floor(
#     (df["TransactionDT"] - df["TransactionDT"].min()) / 86400
# ).astype(int)

# train_period = df[df["day"] <= 121]
# validation_period = df[
#     (df["day"] >= 122) &
#     (df["day"] <= 151)
# ]

# seen_entities = set(train_period["id_combined"].dropna())

# valid_future = validation_period["id_combined"].notna()

# future_seen_before = validation_period.loc[
#     valid_future, "id_combined"
# ].isin(seen_entities)

# print("--- Future Entity Recurrence ---")
# print(f"Validation transactions: {len(validation_period):,}")
# print(
#     f"Transactions with usable entity ID: "
#     f"{valid_future.sum():,} "
#     f"({valid_future.mean() * 100:.2f}%)"
# )

# print(
#     f"Future transactions whose entity appeared previously: "
#     f"{future_seen_before.sum():,} "
#     f"({future_seen_before.mean() * 100:.2f}%)"
# )

--- Future Entity Recurrence ---
Validation transactions: 85,044
Transactions with usable entity ID: 75,958 (89.32%)
Future transactions whose entity appeared previously: 38,311 (50.44%)


In [10]:
# Step 5: Address over-fragmentation and build Strict/Loose IDs
cols = ["TransactionID", "TransactionDT", "isFraud", "addr1", "P_emaildomain", "D1"]
card_cols = [f"card{i}" for i in range(1, 7)]
df = pd.read_csv(transaction_path, usecols=cols + card_cols)

# 1. Check true missing rates for the core components
missing_d1 = df["D1"].isna().mean()
missing_addr1 = df["addr1"].isna().mean()
missing_email = df["P_emaildomain"].isna().mean()

print(f"--- Missing Data Rates ---")
print(f"D1 missing:            {missing_d1:.1%}")
print(f"addr1 missing:         {missing_addr1:.1%}")
print(f"P_emaildomain missing: {missing_email:.1%}\n")

# 2. Create the D1 anchor with a 2-day bucket to tolerate +/- 1 day shifts
df["D1_anchor"] = np.floor((df["TransactionDT"] / 86400) - df["D1"])
df["D1_anchor_bucket"] = np.floor(df["D1_anchor"] / 2) * 2

# 3. Define the Loose ID
df["uid_loose"] = df["card1"].astype(str) + "_" + df["addr1"].astype(str)

# 4. Define the Strict ID (All card features + addr1 + email + bucketed anchor)
# We fill NaNs with a placeholder so string concatenation works,
# but if D1 is missing, we invalidate the whole Strict ID.
strict_cols = card_cols + ["addr1", "P_emaildomain", "D1_anchor_bucket"]
df["uid_strict"] = df[strict_cols].fillna("NA").astype(str).agg('_'.join, axis=1)

# Invalidate Strict ID if the temporal anchor (D1) is missing
df.loc[df["D1"].isna(), "uid_strict"] = np.nan

# Evaluate the new Strict ID using our existing stats function
get_stats(df["uid_strict"], "Candidate 4: Strict ID (Card1-6 + Addr1 + Email + 2-Day D1 Bucket)")

--- Missing Data Rates ---
D1 missing:            0.2%
addr1 missing:         11.1%
P_emaildomain missing: 16.0%

--- Candidate 4: Strict ID (Card1-6 + Addr1 + Email + 2-Day D1 Bucket) ---
Missing records: 1269 (0.21%)
Unique entities: 262,199
Entities with 1 transaction: 166,278 (63.42%)
Entities with >1 transaction: 95,921 (36.58%)
Max transactions by one entity: 1,414
Median transactions per entity: 1.0



## Missing Data Rates and ID Tolerance
D1 is missing in only 0.2% of rows, meaning our temporal anchor is available for almost the entire dataset. addr1 and P_emaildomain are missing at 11% and 16%, but by filling them with a placeholder (like "NA") during ID construction, we successfully group users who consistently leave these fields blank rather than dropping their data

In [11]:
# Create chronological features and test predictive lift with LightGBM
from sklearn.metrics import average_precision_score
import lightgbm as lgb

# 1. Sort data chronologically to prevent future leakage
df = df.sort_values("TransactionDT").copy()

# 2. Create historical features based strictly on past activity
df["strict_prev_count"] = df.groupby("uid_strict").cumcount()
df["strict_time_since_last"] = (
    df.groupby("uid_strict")["TransactionDT"].diff() / 86400
).fillna(-1)

# 3. Prepare train/validation splits (80/20 chronological split)
# We will use just D1 as a baseline feature to keep it simple
features_base = ["D1"]
features_new = ["D1", "strict_prev_count", "strict_time_since_last"]

split_idx = int(len(df) * 0.8)
train, valid = df.iloc[:split_idx], df.iloc[split_idx:]

# 4. Train Baseline Model
model_base = lgb.LGBMClassifier(n_estimators=50, random_state=42, n_jobs=-1, verbose=-1)
model_base.fit(train[features_base], train["isFraud"])
preds_base = model_base.predict_proba(valid[features_base])[:, 1]
auprc_base = average_precision_score(valid["isFraud"], preds_base)

# 5. Train Model with Sequence Features
model_new = lgb.LGBMClassifier(n_estimators=50, random_state=42, n_jobs=-1, verbose=-1)
model_new.fit(train[features_new], train["isFraud"])
preds_new = model_new.predict_proba(valid[features_new])[:, 1]
auprc_new = average_precision_score(valid["isFraud"], preds_new)

print("--- Predictive Validation (AUPRC) ---")
print(f"Baseline AUPRC (D1 only):       {auprc_base:.4f}")
print(f"New AUPRC (with sequence feats): {auprc_new:.4f}")
print(f"Absolute Lift:                  {auprc_new - auprc_base:+.4f}")

--- Predictive Validation (AUPRC) ---
Baseline AUPRC (D1 only):       0.0566
New AUPRC (with sequence feats): 0.0713
Absolute Lift:                  +0.0147


## Testing Predictive Usefulness (AUPRC)
The ultimate test of our proxy ID is whether it helps a model detect fraud. By creating just two simple historical features (previous transaction count and time since last transaction) based on our Strict ID, the AUPRC improved from 0.0566 to 0.0713. This +0.0147 absolute lift on a strictly chronological holdout set proves that our grouping captures real, predictive behavioral sequences without leaking future data

## Final Evidence-Based Conclusion

Verdict: Feasible

Justification for the Team:
The team should proceed with engineering temporal and sequence features.

Empirically Useful: Generating chronological features from our Strict ID (all card features + addr1 + email + bucketed D1 anchor) delivered an immediate AUPRC lift on a chronological validation split.

Missing Data is Manageable: D1 is missing in only 0.2% of records. For addr1 (11% missing) and P_emaildomain (16% missing), treating missing values as a distinct category ("NA") successfully groups repeat users who consistently leave those fields blank.

Tolerance Limits Fragmentation: Bucketing the D1_anchor into 2-day windows mitigates the risk of artificially splitting a single user's session simply because it crossed a midnight boundary. Adding card2-6 and email helps separate unrelated users who share a card1 and addr1.

Caveats and Required Guidelines:

Unvalidated Ground Truth: This is still a proxy. Some entities will inevitably blend users (corporate cards, bots) or fragment a single user (if their D1 gets reset).

Use a Multi-Tier Approach: The team should build features at both a "Strict" level (high precision, lower recall) and a "Loose" level (card1 + addr1). Feed both into the model and let it choose.

Strict Ban on Fraud-Label Features: The team must not build "historical fraud rate" features per ID. In the real world, fraud labels arrive with a significant delay. Using historical labels on this dataset guarantees leakage and will artificially inflate training scores. Stick strictly to behavioral features (time gaps, counts, amount deviations)